# Content Moderation Model Training

This notebook trains multi-label text classification models for detecting harmful content in user-generated text.

**Datasets:**
- Jigsaw Toxic Comment Classification
- ETHOS Multi-label Hate Speech

**Models Compared:**
- BERT (bert-base-uncased)
- RoBERTa (roberta-base)
- DistilBERT (distilbert-base-uncased)

**Labels (Multi-label Classification):**
- Toxicity
- Severe toxicity
- Obscene language
- Threats
- Insults
- Identity hate


In [ ]:
!pip install -q transformers torch datasets scikit-learn pandas numpy joblib tqdm

## 1. Data Loading and Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import hamming_loss, accuracy_score, precision_score, recall_score, f1_score, jaccard_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print("Imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

In [ ]:
from datasets import load_dataset

LABEL_SCHEMA = {
    "toxicity": 0,
    "severe_toxicity": 1,
    "obscene": 2,
    "threat": 3,
    "insult": 4,
    "identity_hate": 5,
}

REVERSE_LABEL_SCHEMA = {v: k for k, v in LABEL_SCHEMA.items()}

def load_jigsaw_dataset():
    """Load Jigsaw Toxic Comment Classification dataset."""
    print("Loading Jigsaw Toxic Comment Classification dataset...")
    dataset = load_dataset("google/jigsaw_toxicity_pred")
    
    df_list = []
    for split in dataset.keys():
        split_df = dataset[split].to_pandas()
        df_list.append(split_df)
    
    df = pd.concat(df_list, ignore_index=True)
    
    if 'comment_text' in df.columns:
        df = df.rename(columns={'comment_text': 'text'})
    
    label_columns = ['toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_hate']
    available_cols = [col for col in label_columns if col in df.columns]
    
    threshold = 0.5
    df['labels'] = df[available_cols].apply(
        lambda row: [LABEL_SCHEMA[col] for col in available_cols if row[col] >= threshold],
        axis=1
    )
    
    df['source'] = 'jigsaw'
    
    return df[['text', 'labels', 'source']].copy()

def load_ethos_dataset():
    """Load ETHOS hate speech dataset with proper multi-label mapping."""
    print("Loading ETHOS hate speech dataset...")
    dataset = load_dataset("iamollas/ethos")
    
    df_list = []
    for split in dataset.keys():
        split_df = dataset[split].to_pandas()
        df_list.append(split_df)
    
    df = pd.concat(df_list, ignore_index=True)
    
    if 'comment' in df.columns:
        df = df.rename(columns={'comment': 'text'})
    
    def map_ethos_labels(row):
        labels = set()

        # Check for identity-hate related labels
        identity_hate_columns = ['gender', 'race', 'national_origin', 'disability', 'religion', 'sexual_orientation']
        has_identity_hate = any(col in row and row[col] == 1 for col in identity_hate_columns)

        if has_identity_hate:
            labels.add(LABEL_SCHEMA['identity_hate'])
            labels.add(LABEL_SCHEMA['toxicity'])

        # Violence maps to threat
        if 'violence' in row and row['violence'] == 1:
            labels.add(LABEL_SCHEMA['threat'])
            labels.add(LABEL_SCHEMA['toxicity'])

        return sorted(list(labels))
    
    df['labels'] = df.apply(map_ethos_labels, axis=1)
    df['source'] = 'ethos'
    
    return df[['text', 'labels', 'source']].copy()

def merge_datasets(jigsaw_df, ethos_df):
    """Merge Jigsaw and ETHOS datasets."""
    print(f"\nMerging datasets: Jigsaw ({len(jigsaw_df)} samples) + ETHOS ({len(ethos_df)} samples)")
    
    merged = pd.concat([jigsaw_df, ethos_df], ignore_index=True)
    merged = merged[merged['text'].notna() & (merged['text'].str.len() > 0)]
    
    merged['label_count'] = merged['labels'].apply(len)
    merged['has_labels'] = merged['label_count'] > 0
    
    print(f"Merged dataset: {len(merged)} samples")
    print(f"  Samples with labels: {merged['has_labels'].sum()}")
    print(f"  Samples without labels: {(~merged['has_labels']).sum()}")
    print(f"\nLabel distribution:")
    
    label_counts = {REVERSE_LABEL_SCHEMA[i]: 0 for i in range(len(LABEL_SCHEMA))}
    for labels_list in merged['labels']:
        for label_idx in labels_list:
            label_counts[REVERSE_LABEL_SCHEMA[label_idx]] += 1
    
    for label_name, count in sorted(label_counts.items()):
        percentage = (count / len(merged)) * 100 if len(merged) > 0 else 0
        print(f"  {label_name}: {count} ({percentage:.2f}%)")
    
    return merged

# Load and merge datasets
jigsaw_df = load_jigsaw_dataset()
ethos_df = load_ethos_dataset()
merged_df = merge_datasets(jigsaw_df, ethos_df)

In [ ]:
from sklearn.model_selection import train_test_split

# Split dataset
test_size = 0.15
val_size = 0.15
random_state = 42

train_val, test_df = train_test_split(
    merged_df,
    test_size=test_size,
    random_state=random_state,
)

val_ratio = val_size / (1 - test_size)
train_df, val_df = train_test_split(
    train_val,
    test_size=val_ratio,
    random_state=random_state,
)

print(f"\nDataset split:")
print(f"  Train: {len(train_df)} samples ({len(train_df)/len(merged_df)*100:.1f}%)")
print(f"  Val:   {len(val_df)} samples ({len(val_df)/len(merged_df)*100:.1f}%)")
print(f"  Test:  {len(test_df)} samples ({len(test_df)/len(merged_df)*100:.1f}%)")

# Show sample
print(f"\nSample text:")
print(f"  Text: {train_df.iloc[0]['text'][:100]}...")
print(f"  Labels: {[REVERSE_LABEL_SCHEMA[i] for i in train_df.iloc[0]['labels']]}")

## 2. Text Preprocessing & Tokenization

In [ ]:
import re

class TextPreprocessor:
    """Text cleaning and normalization."""
    
    @staticmethod
    def clean_text(text):
        if not isinstance(text, str):
            return ""
        
        text = text.strip()
        text = re.sub(r'https?://\S+', '', text)  # URLs
        text = re.sub(r'www\.\S+', '', text)
        text = re.sub(r'<[^>]+>', '', text)  # HTML tags
        text = re.sub(r'@\w+', '', text)  # Mentions
        text = re.sub(r'#\w+', '', text)  # Hashtags
        text = re.sub(r'[^\w\s.,!?-]', '', text)  # Special chars
        text = re.sub(r'\s+', ' ', text).strip()  # Whitespace
        
        return text

def create_label_array(label_indices, num_labels=6):
    """Convert label indices to multi-hot encoded array."""
    array = np.zeros(num_labels, dtype=np.float32)
    for idx in label_indices:
        if 0 <= idx < num_labels:
            array[idx] = 1.0
    return array

# Test preprocessor
preprocessor = TextPreprocessor()
test_text = "This is so TOXIC!!! 🤮 Check out my site: www.spam.com @troll #hate"
cleaned = preprocessor.clean_text(test_text)
print(f"Original: {test_text}")
print(f"Cleaned:  {cleaned}")

In [ ]:
def prepare_data_for_model(df, tokenizer, max_length=512):
    """Prepare texts and labels for model training."""
    preprocessor = TextPreprocessor()
    
    texts = [preprocessor.clean_text(text) for text in df['text'].tolist()]
    labels_array = np.array([create_label_array(labels) for labels in df['labels'].tolist()])
    
    # Tokenize
    encodings = tokenizer(
        texts,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    input_ids = encodings['input_ids']
    attention_masks = encodings['attention_mask']
    labels = torch.tensor(labels_array, dtype=torch.float32)
    
    return input_ids, attention_masks, labels

print("Data preparation function defined.")

## 3. Model Comparison: BERT vs RoBERTa vs DistilBERT

We'll train three models and compare their performance:
- **BERT**: Bidirectional Encoder Representations from Transformers (12 layers)
- **RoBERTa**: Robustly Optimized BERT Pretraining Approach (12 layers, better pretraining)
- **DistilBERT**: Distilled BERT (6 layers, faster but slightly lower performance)

In [ ]:
import time
from tqdm import tqdm

class MultiLabelClassificationModel(torch.nn.Module):
    """Multi-label text classification model."""
    
    def __init__(self, model_name, num_labels=6):
        super().__init__()
        self.transformer = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            problem_type="multi_label_classification"
        )
    
    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        return outputs

def compute_metrics(y_true, y_pred):
    """Compute multi-label classification metrics."""
    binarized_pred = (y_pred > 0.5).astype(int)
    
    return {
        'hamming_loss': hamming_loss(y_true, binarized_pred),
        'jaccard': jaccard_score(y_true, binarized_pred, average='micro', zero_division=0),
        'accuracy': accuracy_score(y_true, binarized_pred),
        'precision': precision_score(y_true, binarized_pred, average='macro', zero_division=0),
        'recall': recall_score(y_true, binarized_pred, average='macro', zero_division=0),
        'f1': f1_score(y_true, binarized_pred, average='macro', zero_division=0),
    }

print("Model and metric functions defined.")

In [ ]:
def train_model(model_name, train_df, val_df, epochs=3, batch_size=16, learning_rate=2e-5):
    """
    Train a multi-label classification model.
    
    Args:
        model_name: HuggingFace model ID
        train_df, val_df: Training and validation DataFrames
        epochs: Number of training epochs
        batch_size: Batch size
        learning_rate: Learning rate
    
    Returns:
        Dictionary with model, tokenizer, metrics, and training history
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"Device: {device}")
    print(f"{'='*60}")
    
    # Load tokenizer and prepare data
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    print("\nPreparing training data...")
    train_input_ids, train_attention_masks, train_labels = prepare_data_for_model(train_df, tokenizer)
    
    print("Preparing validation data...")
    val_input_ids, val_attention_masks, val_labels = prepare_data_for_model(val_df, tokenizer)
    
    # Create data loaders
    train_dataset = TensorDataset(train_input_ids, train_attention_masks, train_labels)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    val_dataset = TensorDataset(val_input_ids, val_attention_masks, val_labels)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    # Initialize model
    model = MultiLabelClassificationModel(model_name, num_labels=6)
    model.to(device)
    
    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    # Training loop
    train_losses = []
    val_metrics_history = []
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        # Training
        model.train()
        train_loss = 0
        progress_bar = tqdm(train_loader, desc="Training")
        
        for batch_idx, (input_ids, attention_mask, labels) in enumerate(progress_bar):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(input_ids, attention_mask, labels=labels)
            loss = outputs.loss
            
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
            progress_bar.set_postfix({'loss': train_loss / (batch_idx + 1)})
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        print(f"Avg Train Loss: {avg_train_loss:.4f}")
        
        # Validation
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            progress_bar = tqdm(val_loader, desc="Validation")
            for input_ids, attention_mask, labels in progress_bar:
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                labels = labels.to(device)
                
                outputs = model(input_ids, attention_mask, labels=labels)
                loss = outputs.loss
                logits = outputs.logits
                
                val_loss += loss.item()
                
                preds = torch.sigmoid(logits).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        print(f"Avg Val Loss: {avg_val_loss:.4f}")
        
        # Compute metrics
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        metrics = compute_metrics(all_labels, all_preds)
        
        print(f"Val Metrics: {', '.join([f'{k}={v:.4f}' for k, v in metrics.items()])}")
        val_metrics_history.append(metrics)
    
    # Final evaluation on test set
    print("\nEvaluating on test set...")
    test_input_ids, test_attention_masks, test_labels = prepare_data_for_model(test_df, tokenizer)
    test_dataset = TensorDataset(test_input_ids, test_attention_masks, test_labels)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    
    model.eval()
    test_loss = 0
    test_preds = []
    test_labels_list = []
    
    with torch.no_grad():
        for input_ids, attention_mask, labels in tqdm(test_loader, desc="Testing"):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            
            outputs = model(input_ids, attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits
            
            test_loss += loss.item()
            
            preds = torch.sigmoid(logits).cpu().numpy()
            test_preds.extend(preds)
            test_labels_list.extend(labels.cpu().numpy())
    
    avg_test_loss = test_loss / len(test_loader)
    test_preds = np.array(test_preds)
    test_labels_array = np.array(test_labels_list)
    test_metrics = compute_metrics(test_labels_array, test_preds)
    
    print(f"\nTest Loss: {avg_test_loss:.4f}")
    print(f"Test Metrics: {', '.join([f'{k}={v:.4f}' for k, v in test_metrics.items()])}")
    
    return {
        'model': model,
        'tokenizer': tokenizer,
        'device': device,
        'train_losses': train_losses,
        'val_metrics': val_metrics_history,
        'test_metrics': test_metrics,
        'test_loss': avg_test_loss,
    }

print("Train function defined.")

In [ ]:
# Train BERT model
bert_result = train_model(
    'bert-base-uncased',
    train_df,
    val_df,
    epochs=3,
    batch_size=16,
    learning_rate=2e-5
)

In [ ]:
# Train RoBERTa model
roberta_result = train_model(
    'roberta-base',
    train_df,
    val_df,
    epochs=3,
    batch_size=16,
    learning_rate=2e-5
)

In [ ]:
# Train DistilBERT model
distilbert_result = train_model(
    'distilbert-base-uncased',
    train_df,
    val_df,
    epochs=3,
    batch_size=32,
    learning_rate=2e-5
)

## 4. Model Evaluation and Comparison

In [ ]:
import matplotlib.pyplot as plt

# Comparison table
comparison_data = {
    'Model': ['BERT', 'RoBERTa', 'DistilBERT'],
    'Test F1': [
        bert_result['test_metrics']['f1'],
        roberta_result['test_metrics']['f1'],
        distilbert_result['test_metrics']['f1'],
    ],
    'Test Precision': [
        bert_result['test_metrics']['precision'],
        roberta_result['test_metrics']['precision'],
        distilbert_result['test_metrics']['precision'],
    ],
    'Test Recall': [
        bert_result['test_metrics']['recall'],
        roberta_result['test_metrics']['recall'],
        distilbert_result['test_metrics']['recall'],
    ],
    'Jaccard': [
        bert_result['test_metrics']['jaccard'],
        roberta_result['test_metrics']['jaccard'],
        distilbert_result['test_metrics']['jaccard'],
    ],
    'Hamming Loss': [
        bert_result['test_metrics']['hamming_loss'],
        roberta_result['test_metrics']['hamming_loss'],
        distilbert_result['test_metrics']['hamming_loss'],
    ],
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*80)
print("MODEL COMPARISON ON TEST SET")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Plot metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

models = ['BERT', 'RoBERTa', 'DistilBERT']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# F1 Score
f1_scores = comparison_df['Test F1'].values
axes[0, 0].bar(models, f1_scores, color=colors)
axes[0, 0].set_ylabel('F1 Score')
axes[0, 0].set_title('F1 Score (Higher is Better)')
axes[0, 0].set_ylim([0, 1])
for i, v in enumerate(f1_scores):
    axes[0, 0].text(i, v + 0.02, f'{v:.4f}', ha='center')

# Precision
precision_scores = comparison_df['Test Precision'].values
axes[0, 1].bar(models, precision_scores, color=colors)
axes[0, 1].set_ylabel('Precision')
axes[0, 1].set_title('Precision (Higher is Better)')
axes[0, 1].set_ylim([0, 1])
for i, v in enumerate(precision_scores):
    axes[0, 1].text(i, v + 0.02, f'{v:.4f}', ha='center')

# Recall
recall_scores = comparison_df['Test Recall'].values
axes[1, 0].bar(models, recall_scores, color=colors)
axes[1, 0].set_ylabel('Recall')
axes[1, 0].set_title('Recall (Higher is Better)')
axes[1, 0].set_ylim([0, 1])
for i, v in enumerate(recall_scores):
    axes[1, 0].text(i, v + 0.02, f'{v:.4f}', ha='center')

# Jaccard
jaccard_scores = comparison_df['Jaccard'].values
axes[1, 1].bar(models, jaccard_scores, color=colors)
axes[1, 1].set_ylabel('Jaccard Index')
axes[1, 1].set_title('Jaccard Index (Higher is Better)')
axes[1, 1].set_ylim([0, 1])
for i, v in enumerate(jaccard_scores):
    axes[1, 1].text(i, v + 0.02, f'{v:.4f}', ha='center')

plt.tight_layout()
plt.show()

print("\n✓ Comparison plots generated")

## 5. Save Models and Configuration

In [ ]:
import json
import joblib
import os

# Create models directory if it doesn't exist
models_dir = './models'
os.makedirs(models_dir, exist_ok=True)

def save_model(result, model_type, model_name):
    """
    Save model, tokenizer, and configuration.
    
    Args:
        result: Training result dictionary
        model_type: Short name (bert, roberta, distilbert)
        model_name: HuggingFace model ID
    """
    model = result['model'].transformer  # Extract transformer from wrapper
    tokenizer = result['tokenizer']
    test_metrics = result['test_metrics']
    
    # Save model
    model_path = os.path.join(models_dir, f'content_moderation_{model_type}.pt')
    torch.save(result['model'], model_path)  # Save the wrapped model
    print(f"✓ Model saved: {model_path}")
    
    # Save config
    config = {
        'model_name': model_name,
        'model_type': model_type,
        'num_labels': 6,
        'test_metrics': test_metrics,
    }
    config_path = os.path.join(models_dir, f'model_config_{model_type}.pkl')
    joblib.dump(config, config_path)
    print(f"✓ Config saved: {config_path}")
    
    # Save label schema
    labels_path = os.path.join(models_dir, 'label_schema.json')
    if not os.path.exists(labels_path):
        with open(labels_path, 'w') as f:
            json.dump({
                'label_schema': LABEL_SCHEMA,
                'reverse_schema': REVERSE_LABEL_SCHEMA,
            }, f, indent=2)
        print(f"✓ Label schema saved: {labels_path}")

# Save all models
save_model(bert_result, 'bert', 'bert-base-uncased')
save_model(roberta_result, 'roberta', 'roberta-base')
save_model(distilbert_result, 'distilbert', 'distilbert-base-uncased')

In [ ]:
# Select the best model based on F1 score
best_model_type = comparison_df.loc[comparison_df['Test F1'].idxmax(), 'Model'].lower()
best_f1 = comparison_df['Test F1'].max()

print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model_type.upper()}")
print(f"F1 Score: {best_f1:.4f}")
print(f"{'='*60}")

# Save best model with special name
if best_model_type == 'bert':
    best_result = bert_result
    best_name = 'bert-base-uncased'
elif best_model_type == 'roberta':
    best_result = roberta_result
    best_name = 'roberta-base'
else:
    best_result = distilbert_result
    best_name = 'distilbert-base-uncased'

save_model(best_result, 'best', best_name)
print(f"\n✓ Best model (type: {best_model_type}) saved as 'best'")

## 6. Test Inference

In [ ]:
def test_inference(model_type='best'):
    """
    Test the saved model with sample texts.
    """
    print(f"\nTesting inference with {model_type} model...\n")
    
    # Load model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_path = os.path.join(models_dir, f'content_moderation_{model_type}.pt')
    model = torch.load(model_path, map_location=device)
    model.eval()
    
    config_path = os.path.join(models_dir, f'model_config_{model_type}.pkl')
    config = joblib.load(config_path)
    tokenizer = AutoTokenizer.from_pretrained(config['model_name'])
    
    # Test samples
    test_samples = [
        "This is a great product! Highly recommend.",
        "You are so stupid and worthless!",
        "I hate people from that country.",
        "This is absolutely disgusting and vile content.",
        "Let's meet up for coffee sometime.",
        "I'm going to beat you up, you piece of trash!",
    ]
    
    preprocessor = TextPreprocessor()
    
    results = []
    with torch.no_grad():
        for text in test_samples:
            cleaned = preprocessor.clean_text(text)
            
            encoding = tokenizer(
                cleaned,
                max_length=512,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            
            input_ids = encoding['input_ids'].to(device)
            attention_mask = encoding['attention_mask'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.sigmoid(logits).cpu().numpy()[0]
            
            detected_labels = []
            scores = {}
            
            for idx, prob in enumerate(probs):
                label_name = REVERSE_LABEL_SCHEMA[idx]
                scores[label_name] = float(prob)
                if prob >= 0.5:
                    detected_labels.append(label_name)
            
            results.append({
                'text': text,
                'labels': detected_labels,
                'scores': scores,
            })
    
    # Print results
    for i, result in enumerate(results, 1):
        print(f"Sample {i}:")
        print(f"  Text: {result['text']}")
        print(f"  Detected Labels: {result['labels'] if result['labels'] else 'None (clean text)'}")
        print(f"  Scores: {', '.join([f'{k}={v:.4f}' for k, v in result['scores'].items()])}")
        print()

test_inference('best')

## 7. Summary and Next Steps

### Training Summary

We have successfully trained three multi-label text classification models:

1. **BERT** - Bidirectional Encoder Representations from Transformers
2. **RoBERTa** - Robustly Optimized BERT Pretraining Approach
3. **DistilBERT** - Distilled BERT (faster, smaller)

### Model Performance

The best model (saved as `content_moderation_best.pt`) has been selected based on F1 score on the test set.

### Deployment

To deploy the models to production:

1. Upload the models from `./models/` directory to the backend:
   ```bash
   gsutil -m cp -r models/* gs://your-bucket/content_moderation/models/
   ```

2. Or copy locally to:
   ```
   /home/capstone/backend/ai_related/content_moderation/machine_learning/models/
   ```

3. The API will automatically load the models on first use.

### API Endpoints

Once deployed, use these endpoints:

- **POST** `/content_moderation/moderate` - Moderate single text
- **POST** `/content_moderation/moderate_batch` - Moderate multiple texts
- **GET** `/content_moderation/labels` - Get label definitions
- **GET** `/content_moderation/models` - Get available models


In [ ]:
print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)
print(f"\nModels saved in: {models_dir}")
print(f"\nFiles created:")
for file in os.listdir(models_dir):
    file_path = os.path.join(models_dir, file)
    file_size = os.path.getsize(file_path) / (1024**2)  # Convert to MB
    print(f"  - {file} ({file_size:.1f} MB)")
print("\n" + "="*80)